In [1]:
import pandas as pd
import sqlite3

In [106]:
connect = sqlite3.connect(":memory:")

In [99]:
# load the dataset using pandas
adm_staff = pd.read_csv("HC_Administrative_Staff.csv")
beds = pd.read_csv("HC_Beds.csv")
billing = pd.read_csv("HC_Billing.csv")
departments = pd.read_csv("HC_Departments.csv")
diagnoses = pd.read_csv("HC_Diagnoses.csv")
districts = pd.read_csv("HC_Districts.csv")
doctors = pd.read_csv("HC_Doctor.csv")
med_equipments = pd.read_csv("HC_Medical_Equipment.csv")
med_specialists = pd.read_csv("HC_Medical_Specialties.csv")
nurses = pd.read_csv("HC_Nurse.csv")
ownership_types = pd.read_csv("HC_Ownership_Types.csv")
patient_feedback = pd.read_csv("HC_Patient_Feedback.csv")
patients = pd.read_csv("HC_Patients.csv")
payments = pd.read_csv("HC_Payments.csv")
provinces = pd.read_csv("HC_Provinces.csv")
radiographer = pd.read_csv("HC_Radiographer.csv")
revenue_summary = pd.read_csv("HC_Revenue_Summary.csv")
rooms = pd.read_csv("HC_Rooms.csv")
sectors = pd.read_csv("HC_Sectors.csv")
services = pd.read_csv("HC_Services.csv")
villages = pd.read_csv("HC_Villages.csv")
countries = pd.read_csv("HCCountries.csv")
insurance_claims = pd.read_csv("Insurance_Claims.csv")
lab_scientists = pd.read_csv("Laboratory_Scientist.csv")
pharmacist = pd.read_csv("Pharmacist.csv")
appointments = pd.read_csv("Appointments.csv")
admissions = pd.read_csv("Admissions.csv")
referrals = pd.read_csv("Referrals.csv")
prescriptions = pd.read_csv("Prescriptions.csv")
lab_tests = pd.read_csv("Laboratory_tests.csv")

In [107]:
adm_staff.to_sql("adm_staff", connect, index=False)
beds.to_sql("beds", connect, index=False)
billing.to_sql("billing", connect, index=False)
departments.to_sql("departments", connect, index=False)
diagnoses.to_sql("diagnoses", connect, index=False)
districts.to_sql("districts", connect, index=False)
doctors.to_sql("doctors", connect, index=False)
med_equipments.to_sql("med_equipments", connect, index=False)
med_specialists.to_sql("med_sqecialists", connect, index=False)
nurses.to_sql("nurses", connect, index=False)
ownership_types.to_sql("ownership_types", connect, index=False)
patient_feedback.to_sql("patient_feedback", connect, index=False)
patients.to_sql("patients", connect, index=False)
payments.to_sql("payments", connect, index=False)
provinces.to_sql("provinces", connect, index=False)
radiographer.to_sql("radiographer", connect, index=False)
revenue_summary.to_sql("revenue_summary", connect, index=False)
rooms.to_sql("rooms", connect, index=False)
sectors.to_sql("sectors", connect, index=False)
services.to_sql("services", connect, index=False)
villages.to_sql("villages", connect, index=False)
countries.to_sql("countries", connect, index=False)
insurance_claims.to_sql("insurance_claims", connect, index=False)
lab_scientists.to_sql("lab_scientists", connect, index=False)
pharmacist.to_sql("pharmacist", connect, index=False)
appointments.to_sql("appointments", connect, index=False)
admissions.to_sql("admissions", connect, index=False)
referrals.to_sql("referrals", connect, index=False)
prescriptions.to_sql("prescriptions", connect, index=False)
lab_tests.to_sql("lab_tests", connect, index=False)

5000

In [77]:
pd.to_datetime(appointments['Date'])

0      2026-01-07
1      2026-01-14
2      2026-04-15
3      2026-08-14
4      2026-05-27
          ...    
4995   2026-04-27
4996   2026-06-09
4997   2026-03-07
4998   2026-02-11
4999   2026-08-20
Name: Date, Length: 5000, dtype: datetime64[ns]

# Patient Analysis

**1. Retrieve all registered patients.**

In [57]:
pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_patients
    FROM patients;
    """,
    connect
)

,total_patients
0,20000


**2. Count patients by gender**

In [51]:
pd.read_sql_query(
    """
    SELECT 
        Gender, COUNT(*) AS patient_count
    FROM patients
    GROUP BY Gender;
    """,
    connect
)

,Gender,patient_count
0,Female,9910
1,Male,10090


**3. Count patients by province**

In [52]:
pd.read_sql_query(
    """
    SELECT
        pr.ProvinceName,
        COUNT(DISTINCT p.PatientID) AS patient_count
    FROM patients p
    JOIN villages v ON p.VillageID = v.VillageID
    JOIN sectors s ON v.SectorID = s.SectorID
    JOIN districts d ON s.DistrictID = d.DistrictID
    JOIN provinces pr ON d.ProvinceID = pr.ProvinceID
    GROUP BY pr.ProvinceName
    ORDER BY patient_count DESC;
    """,
    connect
)

,ProvinceName,patient_count
0,Western,4091
1,Eastern,4067
2,Southern,3967
3,Kigali City,3952
4,Northern,3923


**4. Identify patients above 65 years** 

In [60]:
pd.read_sql_query(
    """
    SELECT COUNT(DISTINCT PatientID) AS "patients > 65 years"
    FROM patients
    WHERE Age > 65;
    """,
    connect
)

,patients > 65 years
0,5487


**5. Find patients with chronic diseases**

In [61]:
pd.read_sql_query(
    """
    SELECT COUNT(DISTINCT PatientID) AS chronic_disease_patients
    FROM patients
    WHERE ChronicCondition IS NOT NULL
          AND TRIM(ChronicCondition) <> '';
    """,
    connect
)

,chronic_disease_patients
0,14929


**6. Retrieve patients covered by insurance**

In [62]:
pd.read_sql_query(
    """
    SELECT 
        COUNT(DISTINCT PatientID) AS insurance_patients
    FROM patients
    WHERE InsuranceID IS NOT NULL
        AND TRIM(InsuranceID) <> '';
    """,
    connect
)

,insurance_patients
0,20000


**7. Count patients by blood group**

In [64]:
pd.read_sql_query(
    """
    SELECT
        BloodGroup, COUNT(DISTINCT PatientID) AS patient_count
    FROM patients
    GROUP BY BloodGroup
    ORDER BY patient_count DESC;
    """,
    connect
)

,BloodGroup,patient_count
0,AB+,2918
1,O-,2896
2,A-,2892
3,A+,2889
4,B+,2833
5,B-,2792
6,O+,2780


**8. List the youngest and oldest patients**

In [71]:
ages_query = """
SELECT
    PatientID, Age
FROM patients
WHERE Age = (SELECT MIN(Age) FROM patients)
    OR Age = (SELECT MAX(Age) FROM patients)
ORDER BY Age DESC;
"""
ages_table = pd.read_sql_query(ages_query, connect)
ages_table

,PatientID,Age
0,PAT00105,90
1,PAT00148,90
2,PAT00204,90
3,PAT00233,90
4,PAT00371,90
...,...,...
464,PAT19399,0
465,PAT19419,0
466,PAT19605,0
467,PAT19660,0


**9. Identify duplicate phone numbers**

In [72]:
pd.read_sql_query(
    """
    SELECT 
        Phone, COUNT(*) AS DuplicateCount
    FROM patients
    WHERE Phone IS NOT NULL
    GROUP BY Phone
    HAVING COUNT(*) > 1
    ORDER BY DuplicateCount DESC;
    """,
    connect
)

,Phone,DuplicateCount
0,2.507880e+11,2288
1,2.507850e+11,2255
2,2.507890e+11,2239
3,2.507870e+11,2225
4,2.507830e+11,2204
5,2.507860e+11,2194
6,2.507820e+11,2187
7,2.507840e+11,2181
8,2.507900e+11,1129
9,2.507810e+11,1098


**10. Calculate average patient age**

In [73]:
pd.read_sql_query(
    """
    SELECT
        ROUND(AVG(Age), 2) AS AveragePatientAge
    FROM patients;
    """,
    connect
)

,AveragePatientAge
0,45.03


## Appointment Analysis

**11. Count appointments by month**

In [82]:
pd.read_sql_query(
    """
    SELECT
        strftime('%m', Date) AS appointment_month,
        COUNT(*) AS appointment_count
    FROM appointments
    GROUP BY appointment_month
    ORDER BY appointment_month;
    """,
    connect
)

,appointment_month,appointment_count
0,01,418
1,02,413
2,03,440
3,04,405
4,05,400
5,06,437
6,07,416
7,08,428
8,09,408
9,10,414


**12. Identify busiest hospitals**

In [83]:
pd.read_sql_query(
    """
    SELECT
        d.FacilityID,
        COUNT(*) AS AppointmentCount
    FROM appointments a
    JOIN departments d ON a.DepartmentID = d.DepartmentID
    GROUP BY d.FacilityID
    ORDER BY AppointmentCount DESC;
    """,
    connect
)

,FacilityID,AppointmentCount
0,HF004,290
1,HF012,269
2,HF018,264
3,HF020,261
4,HF010,261
5,HF007,261
6,HF008,260
7,HF017,253
8,HF016,253
9,HF009,253


**13. Find busiest departments** 

In [85]:
pd.read_sql_query(
    """
    SELECT
        d.DepartmentID,
        d.DepartmentName,
        COUNT(*) AS AppointmentCount
    FROM appointments a
    JOIN departments d ON a.DepartmentID = d.DepartmentID
    GROUP BY d.DepartmentName
    ORDER BY AppointmentCount DESC;
    """,
    connect
)

,DepartmentID,DepartmentName,AppointmentCount
0,DEP170,Cardiology,544
1,DEP176,Radiology,525
2,DEP035,Obstetrics & Gynecology,511
3,DEP017,Laboratory,499
4,DEP191,Emergency,495
5,DEP124,Pediatrics,491
6,DEP018,Pharmacy,490
7,DEP172,Internal Medicine,490
8,DEP063,Surgery,488
9,DEP009,ICU,467


**14. Find doctors with the highest workload**

In [86]:
pd.read_sql_query(
    """
    SELECT
        doc.StaffID,
        doc.FirstName,
        doc.LastName,
        COUNT(a.AppointmentID) AS AppointmentCount
    FROM doctors doc
    JOIN appointments a ON doc.StaffID = a.DoctorID
    GROUP BY doc.StaffID, doc.FirstName, doc.LastName
    ORDER BY AppointmentCount DESC;
    """,
    connect
)

,StaffID,FirstName,LastName,AppointmentCount
0,DO0082,Diane,Mukamana,28
1,DO0144,Emmanuel,Nsengimana,25
2,DO0193,Patrick,Nsengimana,24
3,DO0218,Eric,Niyonzima,24
4,DO0322,Jean,Nsengimana,24
...,...,...,...,...
345,DO0320,Jean,Niyonzima,7
346,DO0149,Aline,Nsengimana,6
347,DO0291,Eric,Uwera,6
348,DO0306,Aline,Nsengimana,6


**15. Identify referral patterns**

In [88]:
pd.read_sql_query(
    """
    SELECT
        source.FacilityID AS ReferringFacilityID,
        r.ReferredFacilityID,
        COUNT(*) AS ReferralCount
    FROM referrals r
    JOIN appointments a ON r.AppointmentID = a.AppointmentID
    JOIN departments source_dept
    ON a.DepartmentID = source_dept.DepartmentID
    JOIN (
        SELECT DISTINCT FacilityID
        FROM departments
    ) source ON source.FacilityID = a.FacilityID
    GROUP BY source.FacilityID, r.ReferredFacilityID
    ORDER BY ReferralCount DESC;
    """,
    connect
)

,ReferringFacilityID,ReferredFacilityID,ReferralCount
0,HF008,HF008,7
1,HF013,HF010,6
2,HF004,HF011,5
3,HF005,HF010,5
4,HF011,HF003,5
...,...,...,...
238,HF020,HF003,1
239,HF020,HF009,1
240,HF020,HF010,1
241,HF020,HF012,1


**16. Count emergency appointments**

In [89]:
pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS EmergencyAppointmentCount
    FROM appointments
    WHERE VisitType = "Emergency";
    """,
    connect
)

,EmergencyAppointmentCount
0,1650


**17. Calculate average appointments per doctor**

In [90]:
pd.read_sql_query(
    """
    SELECT
    ROUND(
        COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT DoctorID), 0), 2
    ) AS AverageAppointmentsPerDoctor
    FROM appointments;
    """,
    connect
)

,AverageAppointmentsPerDoctor
0,14.29


**18. Retrieve appointments by province**

In [91]:
pd.read_sql_query(
    """
    SELECT
        pr.ProvinceName,
        COUNT(a.AppointmentID) AS AppointmentCount
    FROM appointments a
    JOIN patients p ON a.PatientID = p.PatientID
    JOIN villages v ON p.VillageID = v.VillageID
    JOIN sectors s ON v.SectorID = s.SectorID
    JOIN districts d ON s.DistrictID = d.DistrictID
    JOIN provinces pr ON d.ProvinceID = pr.ProvinceID
    GROUP BY pr.ProvinceName
    ORDER BY AppointmentCount DESC;
    """,
    connect
)

,ProvinceName,AppointmentCount
0,Western,1051
1,Northern,1021
2,Eastern,1006
3,Southern,978
4,Kigali City,944


**19. Identify facilities with the highest patient volume**

In [92]:
pd.read_sql_query(
    """
    SELECT
        a.FacilityID,
        COUNT(DISTINCT a.PatientID) AS UniquePatientCount
    FROM appointments a
    GROUP BY a.FacilityID
    ORDER BY UniquePatientCount DESC;
    """,
    connect
)

,FacilityID,UniquePatientCount
0,HF013,277
1,HF014,275
2,HF010,272
3,HF001,272
4,HF011,271
5,HF006,267
6,HF016,260
7,HF008,255
8,HF003,254
9,HF012,247


**20. Find peak consultation periods**

In [94]:
# Unable to find peak consultation periods as no data is provided on consultation hours

## Clinical Analysis

**21. Top diagnoses**

In [108]:
pd.read_sql_query(
    """
    SELECT
        Diagnosis,
        COUNT(*) AS DiagnosisCount
    FROM diagnoses
    GROUP BY Diagnosis
    ORDER BY DiagnosisCount DESC;
    """,
    connect
)

,Diagnosis,DiagnosisCount
0,Gastritis,12208
1,Diabetes,12083
2,Hypertension,12075
3,Malaria,11878
4,Upper Respiratory Infection,11756


**22. Most requested laboratory tests**

In [110]:
pd.read_sql_query(
    """
    SELECT Test, COUNT(*) AS TestCount
    FROM lab_tests
    GROUP BY Test
    ORDER BY TestCount DESC;
    """,
    connect
)

,Test,TestCount
0,Malaria,1272
1,CBC,1257
2,Glucose,1247
3,Urinalysis,1224


**23. Most prescribed medications**

In [111]:
pd.read_sql_query(
    """
    SELECT Medication, COUNT(*) AS PrescriptionCount
    FROM prescriptions
    GROUP BY Medication
    ORDER BY PrescriptionCount DESC;
    """,
    connect
)

,Medication,PrescriptionCount
0,Amlodipine,1068
1,Omeprazole,1040
2,Metformin,999
3,Amoxicillin,979
4,Artemether-Lumefantrine,914


**24. Average admission duration**

In [112]:
pd.read_sql_query(
    """
    SELECT
        ROUND(AVG(LengthOfStayDays), 2) AS AverageLengthOfStayDays
    FROM admissions;
    """,
    connect
)

,AverageLengthOfStayDays
0,5.49


**25. Disease severity distribution**

In [113]:
pd.read_sql_query(
    """
    SELECT
        Severity,
        COUNT(*) AS DiagnosisCount
    FROM diagnoses
    GROUP BY Severity
    ORDER BY DiagnosisCount DESC;
    """,
    connect
)

,Severity,DiagnosisCount
0,Mild,20190
1,Severe,19983
2,Moderate,19827


**26. Laboratory test utilization**

In [114]:
pd.read_sql_query(
    """
    SELECT
        Test,
        COUNT(*) AS TotalTests,
        COUNT(DISTINCT lt.AppointmentID) AS AppointmentsUsingTest
    FROM lab_tests lt
    GROUP BY Test
    ORDER BY TotalTests DESC;
    """,
    connect
)

,Test,TotalTests,AppointmentsUsingTest
0,Malaria,1272,1272
1,CBC,1257,1257
2,Glucose,1247,1247
3,Urinalysis,1224,1224


**27. Prescription trends**

In [116]:
pd.read_sql_query(
    """
    SELECT
        strftime('%m', a.Date) PrescriptionMonth,
        p.Medication,
        COUNT(*) AS PrescriptionCount
    FROM prescriptions p
    JOIN appointments a ON p.AppointmentID = a.AppointmentID
    GROUP BY PrescriptionMonth, p.Medication
    ORDER BY PrescriptionMonth, PrescriptionCount DESC;
    """,
    connect
)

,PrescriptionMonth,Medication,PrescriptionCount
0,01,Amlodipine,75
1,01,Omeprazole,73
2,01,Metformin,67
3,01,Amoxicillin,62
4,01,Artemether-Lumefantrine,57
5,02,Omeprazole,70
6,02,Metformin,67
7,02,Amoxicillin,65
8,02,Artemether-Lumefantrine,57
9,02,Amlodipine,54


**28. Referral destinations**

In [117]:
pd.read_sql_query(
    """
    SELECT
        r.ReferredFacilityID,
        COUNT(*) AS ReferralCount
    FROM referrals r
    WHERE r.ReferredFacilityID IS NOT NULL
    GROUP BY r.ReferredFacilityID
    ORDER BY ReferralCount DESC;
    """,
    connect
)

,ReferredFacilityID,ReferralCount
0,HF014,258
1,HF007,255
2,HF011,250
3,HF001,247
4,HF018,246
5,HF017,246
6,HF010,245
7,HF009,245
8,HF012,241
9,HF008,241


**29. Admission rate by diagnosis**

In [119]:
pd.read_sql_query(
    """
    SELECT
        d.Diagnosis,
        COUNT(DISTINCT ad.AdmissionID) AS AdmissionCount,
        COUNT(DISTINCT d.AppointmentID) AS DiagnosedAppointments,
        ROUND(
            100.0 * COUNT(DISTINCT ad.AdmissionID)
            / NULLIF(COUNT(DISTINCT d.AppointmentID), 0), 2
        ) AS AdmissionRatePercent
    FROM diagnoses d
    LEFT JOIN admissions ad ON d.AppointmentID = ad.AppointmentID
    GROUP BY d.Diagnosis
    ORDER BY AdmissionRatePercent DESC;
    """,
    connect
)

,Diagnosis,AdmissionCount,DiagnosedAppointments,AdmissionRatePercent
0,Upper Respiratory Infection,1012,11756,8.61
1,Hypertension,1017,12075,8.42
2,Diabetes,1008,12083,8.34
3,Gastritis,1002,12208,8.21
4,Malaria,961,11878,8.09


**30. Clinical workload by department**

In [121]:
pd.read_sql_query(
    """
    SELECT
        dep.DepartmentName,
        COUNT(DISTINCT a.AppointmentID) AS AppointmentCount,
        COUNT(DISTINCT dg.DiagnosisID) AS DiagnosisCount,
        COUNT(DISTINCT lt.LabID) AS LaboratoryTestCount,
        COUNT(DISTINCT p.PrescriptionID) AS PrescriptionCount,
        COUNT(DISTINCT ad.AdmissionID) AS AdmissionCount
    FROM departments dep
    LEFT JOIN appointments a ON dep.DepartmentID = a.DepartmentID
    LEFT JOIN diagnoses dg ON a.AppointmentID = dg.AppointmentID
    LEFT JOIN lab_tests lt ON a.AppointmentID = lt.AppointmentID
    LEFT JOIN prescriptions p ON a.AppointmentID = p.AppointmentID
    LEFT JOIN admissions ad ON a.AppointmentID = ad.AppointmentID
    GROUP BY dep.DepartmentID, dep.DepartmentName
    ORDER BY AppointmentCount DESC;
    """,
    connect
)

,DepartmentName,AppointmentCount,DiagnosisCount,LaboratoryTestCount,PrescriptionCount,AdmissionCount
0,Pharmacy,38,38,15,29,3
1,Obstetrics & Gynecology,37,37,11,30,5
2,Emergency,36,36,13,26,5
3,Radiology,35,35,13,27,3
4,Obstetrics & Gynecology,34,34,14,30,9
...,...,...,...,...,...,...
195,Pediatrics,15,15,8,10,0
196,Internal Medicine,15,15,7,14,2
197,Laboratory,15,15,4,13,4
198,ICU,15,15,5,13,2


## Financial Analysis

**31. Revenue by facility** 

In [122]:
pd.read_sql_query(
    """
    SELECT
        a.FacilityID,
        SUM(b.Amount) AS TotalBilledRevenue
    FROM billing b
    JOIN appointments a ON b.AppointmentID = a.AppointmentID
    GROUP BY a.FacilityID
    ORDER BY TotalBilledRevenue DESC;
    """,
    connect
)

,FacilityID,TotalBilledRevenue
0,HF011,44206632
1,HF006,43624779
2,HF013,43547211
3,HF010,42083403
4,HF014,41353389
5,HF001,41131133
6,HF008,39631932
7,HF020,38613123
8,HF003,38561818
9,HF016,38233042


**32. Revenue by department**

In [125]:
pd.read_sql_query(
    """
    SELECT
        d.DepartmentName,
        SUM(b.Amount) AS TotalBilledRevenue
    FROM billing b
    JOIN appointments a ON b.AppointmentID = a.AppointmentID
    JOIN departments d ON a.DepartmentID = d.DepartmentID
    GROUP BY d.DepartmentName
    ORDER BY TotalBilledRevenue DESC;
    """,
    connect
)

,DepartmentName,TotalBilledRevenue
0,Cardiology,86248476
1,Radiology,82000658
2,Laboratory,78562533
3,Pediatrics,77522048
4,Obstetrics & Gynecology,77355750
5,Internal Medicine,75469062
6,Surgery,74420999
7,Emergency,74132460
8,ICU,72934544
9,Pharmacy,72486909


**33. Revenue by service**

In [127]:
pd.read_sql_query(
    """
    SELECT
        s.ServiceName,
        SUM(b.Amount) AS TotalBilledRevenue
    FROM billing b
    JOIN appointments a ON b.AppointmentID = a.AppointmentID
    JOIN services s ON s.DepartmentID = a.DepartmentID
    GROUP BY s.ServiceName
    ORDER BY TotalBilledRevenue DESC;
    """,
    connect
)

,ServiceName,TotalBilledRevenue
0,Cardiology Procedure,86248476
1,Cardiology Consultation,86248476
2,Radiology Procedure,82000658
3,Radiology Consultation,82000658
4,Laboratory Procedure,78562533
5,Laboratory Consultation,78562533
6,Pediatrics Procedure,77522048
7,Pediatrics Consultation,77522048
8,Obstetrics & Gynecology Procedure,77355750
9,Obstetrics & Gynecology Consultation,77355750


**34. Insurance claims by provider**

In [128]:
pd.read_sql_query(
    """
    SELECT
        Provider,
        COUNT(*) AS ClaimCount,
        SUM(ClaimAmount) AS TotalClaimAmount,
        SUM(
            CASE WHEN Status = 'Approved'
            THEN ClaimAmount ELSE 0 END) AS ApprovedClaimAmount,
        SUM(
            CASE WHEN Status = 'Pending'
            THEN ClaimAmount ELSE 0 END) AS PendingClaimAmount
    FROM insurance_claims
    GROUP BY Provider
    ORDER BY TotalClaimAmount DESC;
    """,
    connect
)

,Provider,ClaimCount,TotalClaimAmount,ApprovedClaimAmount,PendingClaimAmount
0,Radiant,10644,1.236512e+09,6.198239e+08,6.166880e+08
1,Private,10508,1.217496e+09,5.952221e+08,6.222743e+08
2,RSSB,10511,1.214649e+09,6.238472e+08,5.908021e+08
3,MMI,10373,1.213206e+09,6.043644e+08,6.088414e+08


**35. Outstanding payments**

In [130]:
pd.read_sql_query(
    """
    SELECT
        SUM(b.Amount - COALESCE(p.AmountPaid, 0)) AS OutstandingAmount
    FROM billing b
    LEFT JOIN (
        SELECT BillingID, SUM(AmountPaid) AS AmountPaid
        FROM payments
        GROUP BY BillingID
    ) p ON b.BillingID = p.BillingID
    WHERE b.Amount - COALESCE(p.AmountPaid, 0) > 0;
    """,
    connect
)

,OutstandingAmount
0,919947405


**36. Average billing amount**

In [131]:
pd.read_sql_query(
    """
    SELECT
        ROUND(AVG(Amount), 2) AS AverageBillingAmount
    FROM billing;
    """,
    connect
)

,AverageBillingAmount
0,155156.7


**37. Monthly revenue trend**

In [132]:
pd.read_sql_query(
    """
    SELECT
        strftime('%m', a.Date) AS RevenueMonth,
        SUM(b.Amount) AS TotalBilledRevenue
    FROM billing b
    JOIN appointments a ON b.AppointmentID = a.AppointmentID
    GROUP BY RevenueMonth
    ORDER BY RevenueMonth;
    """,
    connect
)

,RevenueMonth,TotalBilledRevenue
0,01,63773527
1,02,64318044
2,03,69166031
3,04,63544090
4,05,60802208
5,06,70172522
6,07,64575071
7,08,69091082
8,09,57248879
9,10,63149141


**38. Payment methods**

In [133]:
pd.read_sql_query(
    """
    SELECT
        Method,
        COUNT(*) AS PaymentCount,
        SUM(AmountPaid) AS TotalAmountPaid
    FROM payments
    GROUP BY Method
    ORDER BY TotalAmountPaid DESC;
    """,
    connect
)

,Method,PaymentCount,TotalAmountPaid
0,Card,15145,2115489446
1,Mobile Money,15016,2104779310
2,Insurance,14944,2094324407
3,Cash,14895,2074861393


**39. Top revenue-generating hospitals**

In [135]:
pd.read_sql_query(
    """
    SELECT
    a.FacilityID,
    SUM(b.Amount) AS TotalRevenue
    FROM billing b
    JOIN appointments a ON b.AppointmentID = a.AppointmentID
    GROUP BY a.FacilityID
    ORDER BY TotalRevenue DESC
    LIMIT 10;
    """,
    connect
)

,FacilityID,TotalRevenue
0,HF011,44206632
1,HF006,43624779
2,HF013,43547211
3,HF010,42083403
4,HF014,41353389
5,HF001,41131133
6,HF008,39631932
7,HF020,38613123
8,HF003,38561818
9,HF016,38233042


**40. Financial performance ranking**

In [136]:
pd.read_sql_query(
    """
    SELECT
        a.FacilityID,
        SUM(b.Amount) AS TotalBilledRevenue,
        COALESCE(SUM(p.AmountPaid), 0) AS TotalPaymentsReceived,
        SUM(b.Amount) - COALESCE(SUM(p.AmountPaid), 0) AS OutstandingAmount,
        RANK() OVER (ORDER BY SUM(b.Amount) DESC) AS RevenueRank
    FROM billing b
    JOIN appointments a ON b.AppointmentID = a.AppointmentID
    LEFT JOIN payments p ON b.BillingID = p.BillingID
    GROUP BY a.FacilityID
    ORDER BY RevenueRank;
    """,
    connect
)

,FacilityID,TotalBilledRevenue,TotalPaymentsReceived,OutstandingAmount,RevenueRank
0,HF011,44206632,39810993,4395639,1
1,HF006,43624779,40224830,3399949,2
2,HF013,43547211,39515650,4031561,3
3,HF010,42083403,37969003,4114400,4
4,HF014,41353389,36612652,4740737,5
5,HF001,41131133,37488307,3642826,6
6,HF008,39631932,35661969,3969963,7
7,HF020,38613123,35007484,3605639,8
8,HF003,38561818,34719109,3842709,9
9,HF016,38233042,34389340,3843702,10
